In [25]:
import numpy as np
import pandas as pd
np.random.seed(42)

# CSV-Datei: fehlende Werte, NAN
Es kommt häufig vor, dass in Datensätzen Werte fehlen. Zum Beispiel fehlende Sensorwerte oder nicht als Zahl intertpretierbare Zahlen. 

## Was sind NAN-Werte?
NaN-Werte (kurz für Not a Number) sind spezielle Werte in Daten, die verwendet werden, um fehlende oder undefinierte Daten in einem Datensatz darzustellen. Sie sind ein zentraler Bestandteil von Datenanalyse- und Datenverarbeitungsbibliotheken wie Pandas. 

Numpy bietet einen numerischen Datentyp `nan` an, den wir nutzen können. Im folgenden Beispiel erstellen wir einen DataFrame aus einem Dictionary mit einem NaN-Wert. Uns fällt auf, dass der Wert NaN ein Float-Wert ist (nach ISO 754), analog zu Python.

In [26]:
type(np.nan), type(float("nan"))

(float, float)

In [27]:
# erstelle ein Dict mit NAN-Werten
d = {
    "a": [1, 2, 3, 4],
    "b": [4, 5, np.nan, 6],
}

features = pd.DataFrame(d)
print("fA: \n ", features)
print()
print("fB: \n", features.b)
print()
print("type a: ", features.a.dtype)
print("type b: ", features.b.dtype, "-float es debido al NaN-")


fA: 
     a    b
0  1  4.0
1  2  5.0
2  3  NaN
3  4  6.0

fB: 
 0    4.0
1    5.0
2    NaN
3    6.0
Name: b, dtype: float64

type a:  int64
type b:  float64 -float es debido al NaN-


### Probleme beim arithmetischen Operationen
Bei vielen Daten würde es u.U. gar nicht auffallen, dass sich NaN-Werte eingeschlichen hätten. Würden wir zum Beispiel die Summe aller Werte einer Spalte berechnen, wäre das Ergebnis allerdings kein NaN-Wert, sondern der NaN-Wert wird einfach ignoriert! 

Generell gilt: bei arithmetischen Operationen wie `sum()` werden NaN-Werte als 0 aufgefasst.

In [28]:
# wir können zwar damit rechnen, Nan-Werte werden als 0 aufgefasst

#### Cumsum
in der kummulierten Summe ergibt sich ein falscher Wert, da NaN-Werte ignoriert werden. 

Mit `skipna` lässt sich dieses Verhalten ändern. Dann findet tatsächlich eine Addition statt, aber es gilt: 2 + Nan = NaN.

In [29]:
# cumsum von Spalte b
features.b.cumsum()

0     4.0
1     9.0
2     NaN
3    15.0
Name: b, dtype: float64

## Beispiel-Datei Sensordaten
Wir wollen Sensordaten einer sensordata.csv importieren und prüfen, wie wir mit fehlenden Werten umgehen können.

### Datei einlesen
Probleme: Nicht vorhande Werte werden als NaN (Not a number) abgebildet und die Werte der ersten Zeile der kopflosen CSV-Datei werden als Spaltennamen verwendet. Wenn wir beim Einlesen `header=None` setzen, werden numerische Spaltennamen vergeben (Int64Index).

In [32]:
file_name = "../data/sensordata.csv"
df = pd.read_csv(file_name, header=None)
df.head(3)

,0,1,2
0,NaN,1218.00,1210.18
1,NaN,NaN,868.44
2,1117.92,636.51,NaN


### 1. Spaltennamen anpassen
Die 3 Spalten bezeichnen die 3 Sensoren A, B, C. Wir wollen die Spaltennamen jetzt umbenennen.

In [36]:
# Spaltennamen angeben
df.columns = ["A", "B", "C"]
df.head(3)

,A,B,C
0,NaN,1218.00,1210.18
1,NaN,NaN,868.44
2,1117.92,636.51,NaN


## Fehlende Werte anzeigen lassen
Wir können uns eine boolsche Matrix mit den fehlenden Werten anzeigen lassen. Mit solchen boolschen Serien und Matritzen lässt sich später auch filtern.

In [55]:
# isna nutzen, umd eine boolsche Matrix zu erstellen
missing = df.isna()
missing.head(10)

,A,B,C
0,True,False,False
1,True,True,False
2,False,False,True
3,False,False,True
4,False,False,False
5,False,False,False
6,False,False,False
7,True,False,False
8,False,False,True
9,False,False,False


### Fehlende Werte zählen
- isna(): Not Available
- isnull(): ältere Methode, die genau gleich isna() ist (deprecated)

In [38]:
# Spaltenweise NAN-Werte zählen
df.isna().sum()

A    20
B    33
C    39
dtype: int64

In [39]:
df.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 1000 entries, 0 to 999
Data columns (total 3 columns):
 #   Column  Non-Null Count  Dtype  
---  ------  --------------  -----  
 0   A       980 non-null    float64
 1   B       967 non-null    float64
 2   C       961 non-null    float64
dtypes: float64(3)
memory usage: 23.6 KB


### Prüfen, ob in Spalte A ein wahrer Wert vorkommt (Series Methoden)

Die Methoden any, all und isnull() können ausgefüht werden, zu prüfen, ob Werte falsy oder NaN sind
- isnull: eine boolsche Serie, die wahr zeigt, wenn NaN
- any: eine boolsche Funktion, die wahr ist, wenn ein Wert der Serie wahr ist
- all: eine boolsche Funktion, die wahr ist, wenn ein Wert der Serie wahr ist

In [46]:
# any nutzen, um zu prüfen, ob in Spalte a zumindest ein wahrer Wert vorkommt (vlg. Python any)
d = {
    "a": [0, 0, 0, np.nan],
    "b": [4, 5, 1, 6],
    "c": [0, 0, 0, 0]
}
features = pd.DataFrame(d)
print("hay en Col-A al menos un True?: ", features.a.any())  # es False por que no hay nada diferente a 0 (cero) o NaN
print("hay en Col-B al menos un True?: ", features.b.any())  # es True por que existe al menos 1 valos > a 0 (cero)
print("hay en Col-C al menos un True?: ", features.c.any())


hay en Col-A al menos un True?:  False
hay en Col-B al menos un True?:  True
hay en Col-C al menos un True?:  False

existe un Null-valor en Col-A?: 
  0    False
1    False
2    False
3     True
Name: a, dtype: bool
--------------
existe un Null-valor en Col-B?: 
 0    False
1    False
2    False
3    False
Name: b, dtype: bool
--------------
existe un Null-valor en Col-C?: 
 0    False
1    False
2    False
3    False
Name: c, dtype: bool


### Prüfen, ob in Spalte A alle Werte truthy sind (all)

In [ ]:
# all nutzen
print("existe un Null-valor en Col-A?: \n ", features.a.isna())
print("--------------")
print("existe un Null-valor en Col-B?: \n", features.b.isna())
print("--------------")
print("existe un Null-valor en Col-C?: \n", features.c.isna())

### 2. Zeilen mit Nan-Werten aus dem DataFrame löschen
eine mögliche Strategie, mit NaN-Werten umzugehen, ist es, Zeilen, die mindestens einen NaN-Wert enthalten, komplett zu löschen. Das ist nicht immer möglich aber bei Sensordaten, die im Millisekundentakt gemessen werden, ist das eine angemessene Strategie.

In [56]:
# DataFrame anhand der Rows-Achse löschen, wenn ein NAN-Wert in der Zeile vorkommt
df_cleaned = df.dropna()
print(df_cleaned.head(15))
# se elimino la fila 7 y 8 por q tenian NaN

          A        B        C
4   2043.03  1103.48  1143.08
5   2081.55   662.27  1799.80
6    276.57  1324.42  1063.86
9   1266.55  1410.51  1803.36
10  1582.38  1281.72  1727.09
11  1956.28  1664.06  1835.61
12  1485.63  1467.37  1099.24
13  1693.10   215.69   948.56
14   533.77   632.79  1888.54
15   660.70  1757.34   649.42
16   131.84   936.85  1622.84
17  1668.30   226.27  1049.53
18  1695.05  1064.42   749.30
19   602.67   147.48  1387.60
20  2093.14  1398.45  1263.29


### 3. Mindest Anzahl an NaN-Werten pro Zeile
wir können für die Methode `dropna()` auch einen threshhold angeben, ab wievielen Nan-Werten pro Zeile die Zeile gelöscht werden soll. Im folgenden Beispiel löschen wir nur Zeilen aus dem Dataframe, die mindestens zwei NaN-Werte enthalten.

In [66]:
# threshold angeben, wieviele NaN-Werte in der Zeile vorkommen müssen

df.dropna(axis="index", thresh=2).head(10) # -THRESH- cuántos valores NO-NaN (válidos) debe tener la fila para NO borrarse


,A,B,C
0,0.00,1218.00,1210.18
1,0.00,0.00,868.44
2,1117.92,636.51,0.00
3,922.26,1400.81,0.00
4,2043.03,1103.48,1143.08
5,2081.55,662.27,1799.80
6,276.57,1324.42,1063.86
7,0.00,881.55,2054.76
8,1213.44,1845.42,0.00
9,1266.55,1410.51,1803.36


### Spalten löschen
bisher haben wir immer ganze Spalten gelöscht, wenn ein NaN-Wert in der Zeile vorkam. Wir können natürlich auch Spalten löschen. Im Beispiel bleibt nur der Index übrig, weil jede Spalte mindestens einen Nan-Wert besitzt. Grundsätzlich ist das Löschen ganzer Spalten wohl eher die Ausnahme.

In [ ]:
# Zeilen löschen

### Inplace Löschen
wir müssen den Dataframe nicht zwingend umkopieren, es ist auch ein inplace-Löschen der Zeilen möglich

In [ ]:
# Zeilen inplace llöschen

## Fehlende Werte auffüllen
Wir können fehlende Werte auch mit Platzhalterwerten auffüllen. Im folgenden Beispiel füllen wir alle fehlenden Werte mit 0 auf. Die Operation lässt sich nütürlich ebenso inplace durchführen.

In [65]:
# Fehlende Werte auffüllen
df.fillna(value=0, inplace=True) # value = df.mean(), value=df.A.mean()
df.head()

,A,B,C
0,0.00,1218.00,1210.18
1,0.00,0.00,868.44
2,1117.92,636.51,0.00
3,922.26,1400.81,0.00
4,2043.03,1103.48,1143.08


## Aufgabe: Player Daten

- Lade die CSV-Datei "player_data.csv" in einen Pandas DataFrame. Es sollen im weiteren nur die Spalten id", "playername", "points", "power" berücksichtigt werden (siehe usecols)
- Lösche alle Zeilen, in denen "points" oder "power" fehlen.
- Forme Float-Spalten zu Int8-Spalten um
- ändere Spaltenname: playername zu name
- id wird der neue Index (lösche dazu auch die Spalte id)
- Speichere das bereinigte Ergebnis in einem neuen DataFrame df_clean.
- Gib die ersten 3 Zeilen von df_clean aus.

In [ ]:
import intro_pandas as pd
import numpy as np
# CSV-Datei speichern
csv_path = "../data/player_data.csv"


In [ ]:
# Lese CSV ein und berückichtige alle bis auf die Spalte temp

# drop Einträge, in denen entweder in points oder power ein nan steht

# weise neuen Index zu

# droppe alte Spalte id

# Rename Spalte

# Ändere die Datentypen der Spalten points und power nach int8

# Gebe die ersten 3 Zeilen aus


## Fehlende Werte interpolieren
Wir können fehlende Werte natürlich auch durch andere, selbst errechnete Werte ersetzen, dazu später mehr. Eine einfache Methode, fehlende Werte mit Mittelwerten vorhergehender und nachfolgender Werte zu ersetzen, ist die Methode `interpolate`. Mit dem Parameter `limit_direction` lässt sich bestimmen, ob die Interpolation von "vorne" oder hinten durchgeführt wird. Der default-Wert ist hier `forward`.

Zusätzlich kann auch noch mit `method` die Interpolations-Methode gewählt werden. Neben `linear` (default-Wert) gibt es noch `nearest`, `values`, `quadratic` und einige andere mehr. Hier sei wieder auf die Doku verwiesen:

https://pandas.pydata.org/pandas-docs/version/0.25.0/reference/api/pandas.DataFrame.interpolate.html?highlight=interpolate#pandas.DataFrame.interpolate

In [ ]:
# Fehlende Werte in  Spalten interpolieren

In [ ]:
# Da hier die ersten beiden Werte der Spalte NaN sind, lässt sich mit forward nicht interpolieren.
# dann kann backward genutzt werden, um von hinten zu beginnen